# Homework: Evaluation

In [6]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
	repo_owner="DataTalksClub",
	repo_name="llm-zoomcamp",
	commit_id="8c1834d",
	allowed_extensions={"md"},
	filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

## Q1. Generating questions

In [7]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [8]:
from pydantic import BaseModel

class Questions(BaseModel):
	questions: list[str]

In [9]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [15]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
openai_client = OpenAI()

In [11]:
from evaluation_utils import llm_structured_retry
import json

In [18]:
def generate_ground_truth(doc):
	user_prompt = json.dumps(doc)

	out, usage = llm_structured_retry(
		openai_client,
		data_gen_instructions,
		user_prompt,
		Questions
	)

	results = []

	for q in out.questions:
		results.append({
			"question": q,
			"document": doc["filename"]
		})

	return results, usage

In [20]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:3]):
	records, usage = generate_ground_truth(doc)
	ground_truth.extend(records)
	usages.append(usage)

  0%|          | 0/3 [00:00<?, ?it/s]

In [26]:
input_tokens = [usage.input_tokens for usage in usages]
avg_input_tokens = sum(input_tokens) / len(input_tokens)

In [27]:
avg_input_tokens

1353.0

## Q2. First result with text search

In [59]:
import pandas as pd
df_ground_truth = pd.read_csv('./ground-truth.csv')
ground_truth = df_ground_truth.to_dict(orient="records")

In [60]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [61]:
from minsearch import Index

text_index = Index(
	text_fields=['content'],
	keyword_fields=['filename']
)

text_index.fit(chunks)

In [62]:
def text_search(query, num_results=5):
	return text_index.search(query, num_results=num_results)

## Q3. First result with vector search

In [63]:
from embedder import Embedder
embedder = Embedder()

In [64]:
import numpy as np
X = np.array(embedder.encode_batch([chunk['content'] for chunk in chunks]))

In [65]:
from minsearch import VectorSearch

vector_index = VectorSearch(keyword_fields=["filename"])
vector_index.fit(X, chunks)

In [66]:
def vector_search(query, num_results=5):
  query_vector = embedder.encode(query)
  return vector_index.search(query_vector, num_results=num_results)

### Hybrid search

In [67]:
def rrf(result_lists, k=60, num_results=5):
	scores = {}
	docs = {}

	for results in result_lists:
		for rank, doc in enumerate(results):
			key = (doc["filename"], doc["start"])
			scores[key] = scores.get(key, 0) + 1 / (k + rank)
			docs[key] = doc

	ranked = sorted(scores, key=scores.get, reverse=True)
	return [docs[key] for key in ranked[:num_results]]

In [68]:
def hybrid_search(query, k=60):
	text_results = text_search(query, num_results=10)
	vector_results = vector_search(query, num_results=10)
	return rrf([text_results, vector_results], k=k)

In [69]:
q = ground_truth[0]["question"]

In [96]:
text_search_results = text_search(q)
text_search_results[0]['filename']

'01-agentic-rag/lessons/03-rag.md'

In [72]:
vector_search_results = vector_search(q)
vector_search_results[0]['filename']

'01-agentic-rag/lessons/01-intro.md'

## Q4. Evaluating text search

In [77]:
def compute_relevance(q, search_function):
	filename = q["filename"]
	results = search_function(query=q["question"])

	relevance = []
	for d in results:
		relevance.append(int(d["filename"] == filename))

	return relevance

In [78]:
def compute_relevance_total(ground_truth, search_function):
	relevance_total = []

	for q in tqdm(ground_truth):
		relevance = compute_relevance(q, search_function)
		relevance_total.append(relevance)

	return relevance_total

In [79]:
def hit_rate(relevance):
	cnt = 0

	for line in relevance:
		if 1 in line:
			cnt = cnt + 1

	return cnt / len(relevance)

In [86]:
text_search_relevance = compute_relevance_total(ground_truth, text_search)
text_search_hit_rate = hit_rate(text_search_relevance)

  0%|          | 0/360 [00:00<?, ?it/s]

In [87]:
round(text_search_hit_rate, 2)

0.76

## Q5. Evaluating vector search

In [ ]:
def mrr(relevance):
	total_score = 0.0

	for line in relevance:
		for rank in range(len(line)):
			if line[rank] == 1:
				total_score = total_score + 1 / (rank + 1)
				break

	return total_score / len(relevance)

In [91]:
vector_search_relevance = compute_relevance_total(ground_truth, vector_search)
vector_search_mrr = mrr(vector_search_relevance)

  0%|          | 0/360 [00:00<?, ?it/s]

In [92]:
round(vector_search_mrr, 2)

0.55

## Q6. Tuning hybrid search

In [93]:
k_vals = [1, 50, 100, 200]

In [95]:
for k in k_vals:
  hybrid_search_relevance = compute_relevance_total(ground_truth, lambda query: hybrid_search(query=query, k=k))
  hybrid_search_mrr = mrr(hybrid_search_relevance)
  print(f"k = {k} | MRR = {round(hybrid_search_mrr, 2)}")

  0%|          | 0/360 [00:00<?, ?it/s]

k = 1 | MRR = 0.65


  0%|          | 0/360 [00:00<?, ?it/s]

k = 50 | MRR = 0.64


  0%|          | 0/360 [00:00<?, ?it/s]

k = 100 | MRR = 0.64


  0%|          | 0/360 [00:00<?, ?it/s]

k = 200 | MRR = 0.64
